# Example - Choice of Treatment (Li and Pearl, 2024)

In this notebook, the results of the "Choice of Treatment" example are replicated.

References:

> **"Probabilities of Causation with Nonbinary Treatment and Effect"**
> 
> https://ojs.aaai.org/index.php/AAAI/article/view/30030


> **"Probabilities of Causation with Nonbinary Treatment and Effect"**
> 
> arXiv:2208.09568v1
> 
> https://arxiv.org/pdf/2208.09568

In [1]:
import sys
sys.path.insert(0, '../src/')

import numpy as np

from probabilities_of_causation import *

## Treatments and outcomes 

In [2]:
treatment = {'x1': 'surgery', 'x2': 'chemotherapy', 'x3': 'radiation'}
outcome = {'y1': 'ineffective', 'y2': 'cured', 'y3': 'death'} 

## Experimental and observational data

See paper `Table 1: Experimental data collected by the hospital`. 

|         | x1: Surgery | x2: Chemotherapy | x3: Radiation |
|---------|---------|--------------|-----------|
| y1: Ineffective | 80 | 184| 87 |
| y2: Cured | 7 | 29 | 189|
| y3: Death | 213 | 87 | 24 |
| Overall | 300 | 300 | 300|

In [3]:
# tuple format (y, x)
experimental_data = {
    ('y1', 'x1'): 80/300, 
    ('y2', 'x1'): 7/300, 
    ('y3', 'x1'): 213/300,
    ('y1', 'x2'): 184/300, 
    ('y2', 'x2'): 29/300, 
    ('y3', 'x2'): 87/300,
    ('y1',  'x3'): 87/300, 
    ('y2',  'x3'): 189/300, 
    ('y3',  'x3'): 24/300,
}

See paper `Table 2: Observational data collected by the hospital`. 

|         | x1: Surgery | x2: Chemotherapy | x3: Radiation |
|---------|---------|--------------|-----------|
| y1: Ineffective | 238 | 10 | 147 |
| y2: Cured | 20 | 77 | 72 |
| y3: Death | 7 | 259 | 70 |
| Overall | 265 | 346 | 289 |

In [4]:
# tuple format (x, y)  [NOTE: order inverted w.r.t experimental! ]
observational_data = {
     ('x1', 'y1'): 238/900,  
        ('x1', 'y2'): 20/900,   
        ('x1', 'y3'): 7/900,    
        ('x2', 'y1'): 10/900,  
        ('x2', 'y2'): 77/900,  
        ('x2', 'y3'): 259/900, 
        ('x3', 'y1'): 147/900,  
        ('x3', 'y2'): 72/900,  
        ('x3', 'y3'): 70/900   
}

In [5]:
probabilities = Probabilities(observational_data, experimental_data)


=== Data Completeness Check ===

✓ All observational probabilities P(x, y) are present

✓ All experimental probabilities P(y|do(x)) are present

=== Marginal Probability Check ===
P(x1) = 0.2944
P(x2) = 0.3844
P(x3) = 0.3211
P(y1) = 0.4389
P(y2) = 0.1878
P(y3) = 0.3733
=== End of Data Completeness Check ===



## Query

The patient desires the probability of causation $P({y_3}_{x_1},{y_1}_{x_2},{y_2}_{x_3})$. That is:
- the probability that the patient would would die ($y_3$)  if he chose surgery ($x_1$)
- the probability that nothing would change ($y_1$) if the patient chose chemotherapy ($x_2$)
- the probability that the patient would be cured ($y_2$) if he chose radiation($x_3$)


In [6]:
query = {
    "hypotheticals": [('y3', 'x1'), ('y1', 'x2'),('y2', 'x3')],
    "x_p": None,
    "y_q": None
}

In [7]:
lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 8: probability of necessity and sufficiency
Lower bound: 0.000
Upper bound: 0.099 


Alternatively, theorem 8 can be invoqued directly:

In [8]:
hypotheticals = query.get("hypotheticals", [])
lb, ub = prob_necessity_sufficiency_k(hypotheticals, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Lower bound: 0.000
Upper bound: 0.099 


## Verify the obtained bounds

According to the paper:

$0\le P({y_3}_{x_1},{y_1}_{x_2},{y_2}_{x_3}) \le 0.099$

In [9]:
np.isclose(0, lb, atol=1e-3)

np.True_

In [10]:
np.isclose(0.099, ub, atol=1e-3)

np.True_

##  Intermediate calculation steps reported in the paper
See `Appendix/Calculation in the Examples/Choice of Treatment`

> https://arxiv.org/pdf/2208.09568

### By Theorem 8:

$\large  0.323 \le P({y_3}_{x_1},{y_1}_{x_2})\le 0.340$

In [11]:
query = {
    "hypotheticals": [('y3', 'x1'), ('y1', 'x2')],
    "x_p": None,
    "y_q": None
}

lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 8: probability of necessity and sufficiency
Lower bound: 0.323
Upper bound: 0.340 


### By Theorem 8:
    
$\large 0.243 \le P({y_1}_{x_2},{y_2}_{x_3})\le 0.386 $

In [12]:
query = {
    "hypotheticals": [('y1', 'x2'), ('y2', 'x3')],
    "x_p": None,
    "y_q": None
}

lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 8: probability of necessity and sufficiency
Lower bound: 0.243
Upper bound: 0.386 


### By Theorem 8:
    
$\large 0.340 \le P({y_3}_{x_1},{y_2}_{x_3})\le 0.472$

In [13]:
query = {
    "hypotheticals": [('y3', 'x1'), ('y2', 'x3')],
    "x_p": None,
    "y_q": None
}

lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 8: probability of necessity and sufficiency
Lower bound: 0.340
Upper bound: 0.472 


### By Theorem 11:

$\large 0 \le P(x_1,y_3,{y_1}_{x_2},{y_2}_{x_3})\le 0.008$

In [14]:
query = {
    "hypotheticals": [('y1', 'x2'), ('y2', 'x3')],
    "x_p": 'x1',
    "y_q": 'y3'
}

lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 11: probability of necessity
Lower bound: 0.000
Upper bound: 0.008 


### By Theorem 11:

$\large 0 \le P(x_2,y_1,{y_3}_{x_1},{y_2}_{x_3})\le 0.011$

In [15]:
query = {
    "hypotheticals": [('y3', 'x1'), ('y2', 'x3')],
    "x_p": 'x2',
    "y_q": 'y1'
}

lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 11: probability of necessity
Lower bound: 0.000
Upper bound: 0.011 


### By Theorem 11:

$\large 0 \le P(x_3,y_2,{y_3}_{x_1},{y_1}_{x_2})\le 0.080$

In [16]:
query = {
    "hypotheticals": [('y3', 'x1'), ('y1', 'x2')],
    "x_p": 'x3',
    "y_q": 'y2'
}

lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 11: probability of necessity
Lower bound: 0.000
Upper bound: 0.080 
